# RotaAI — YOLO26 Sign Detector Training (Colab GPU)
Phase 0 baseline. Set **Runtime → Change runtime type → GPU**, then Run all.
PRD: `docs/02-ai-pipeline.md`.

In [ ]:
# 1. Verify GPU
!nvidia-smi

In [ ]:
# 2. Install Ultralytics (provides YOLO26)
!pip -q install 'ultralytics>=8.3.0'
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Get a PUBLIC real-street traffic-sign dataset (YOLO format).
# Example: Roboflow Universe. Replace with your chosen project/version.
#   !pip -q install roboflow
#   from roboflow import Roboflow
#   rf = Roboflow(api_key='YOUR_KEY')
#   ds = rf.workspace('WS').project('PROJECT').version(1).download('yolov8', location='data')
#   DATA_YAML = ds.location + '/data.yaml'
DATA_YAML = 'data/data.yaml'  # <-- point at the downloaded dataset's data.yaml
print('Using', DATA_YAML)

In [ ]:
# 4. Train YOLO26-small (fallback yolo11s.pt if the weight is unavailable)
from ultralytics import YOLO
try:
    model = YOLO('yolo26s.pt')
except Exception as e:
    print('yolo26s unavailable, falling back to yolo11s:', e)
    model = YOLO('yolo11s.pt')
model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16, device=0,
            project='runs', name='rotaai_signs_v0', patience=20)

In [ ]:
# 5. Evaluate → precision / recall / mAP (the accuracy report)
best = 'runs/rotaai_signs_v0/weights/best.pt'
m = YOLO(best).val(data=DATA_YAML)
print({'precision': float(m.box.mp), 'recall': float(m.box.mr),
       'mAP50': float(m.box.map50), 'mAP50_95': float(m.box.map)})

In [ ]:
# 6. Download the trained weights back to your machine
from google.colab import files
files.download(best)